### 鸢尾花案例学习

In [1]:
from sklearn.datasets import load_iris

iris = load_iris()

In [2]:
# 特征 = x，标签 = y
X = iris.data
y = iris.target

print("特征(X) 形状:", X.shape)
print("特征(X) 前10个数据:\n", X[:10])

print("\n标签(y) 形状:", y.shape)
print("标签(y) 前10个数据:\n", y[:10])

特征(X) 形状: (150, 4)
特征(X) 前10个数据:
 [[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]
 [5.4 3.9 1.7 0.4]
 [4.6 3.4 1.4 0.3]
 [5.  3.4 1.5 0.2]
 [4.4 2.9 1.4 0.2]
 [4.9 3.1 1.5 0.1]]

标签(y) 形状: (150,)
标签(y) 前10个数据:
 [0 0 0 0 0 0 0 0 0 0]


In [3]:
# 查看数据基本信息
print("数据集描述：\n", iris['DESCR'][:500], '...')  # 只显示前500个字符，避免输出过长
print("\n特征名称:", iris.feature_names)
print("标签类别:", iris.target_names)

数据集描述：
 .. _iris_dataset:

Iris plants dataset
--------------------

**Data Set Characteristics:**

:Number of Instances: 150 (50 in each of three classes)
:Number of Attributes: 4 numeric, predictive attributes and the class
:Attribute Information:
    - sepal length in cm
    - sepal width in cm
    - petal length in cm
    - petal width in cm
    - class:
            - Iris-Setosa
            - Iris-Versicolour
            - Iris-Virginica

:Summary Statistics:

============== ==== ==== ======= ===== ...

特征名称: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
标签类别: ['setosa' 'versicolor' 'virginica']


#### train_test_split

In [4]:
from sklearn.model_selection import train_test_split

# 划分数据集：训练集70%，测试集30%，随机种子42
# 去除 stratify=y 参数可以让最后knn准确率达到100%
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("训练集X形状:", X_train.shape)
print("测试集X形状:", X_test.shape)
print("训练集y形状:", y_train.shape)
print("测试集y形状:", y_test.shape)

训练集X形状: (105, 4)
测试集X形状: (45, 4)
训练集y形状: (105,)
测试集y形状: (45,)


In [5]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. 创建K近邻分类器，选择k=3
knn = KNeighborsClassifier(n_neighbors=3)

# 2. 在训练集上训练模型
knn.fit(X_train, y_train)

# 3. 在测试集上进行预测
y_pred = knn.predict(X_test)

# 4. 评估模型性能
acc = accuracy_score(y_test, y_pred)
print("K近邻算法在测试集上的准确率: {:.2f}%".format(acc * 100))

# 为什么准确率不是100%？
# 原因如下：
# 1. 训练集与测试集的数据不同，模型没有见过测试集的数据，因此很难做到完全正确预测（尤其是数据有噪声、分布有差异等）。
# 2. K近邻算法属于惰性学习，它依赖于“距离”判断所属类别，如果特征空间中不同类别存在重叠或分布较近，容易导致误判。
# 3. Iris数据集虽然很规范，但类别之间还是存在一些边界模糊的数据点，KNN容易在边界处判断错误。
# 4. 模型参数（如k的选择、未做特征缩放等）也会影响KNN对测试集的泛化能力。
# 总结：准确率不是100%是正常现象，这也反映了机器学习模型对未知数据的泛化能力有限，往往不能完全复现训练集的效果。

K近邻算法在测试集上的准确率: 95.56%


### DictVectorizer

In [6]:
from sklearn.feature_extraction import DictVectorizer

# 字典向量化

data = [
    {'颜色': '红', '尺寸': '大', '价格': 100},
    {'颜色': '蓝', '尺寸': '中', '价格': 80},
    {'颜色': '红', '尺寸': '小', '价格': 50}
]

# 创建DictVectorizer对象
# sparse=False 表示输出为密集数组，而不是稀疏矩阵
vec = DictVectorizer(sparse=False)

# 对data进行fit_transform处理
data_vec = vec.fit_transform(data)

print("特征名字:", vec.get_feature_names_out())
print("转换后的数据:\n", data_vec)

# one-hot编码是一种常用的特征离散化方法，主要用于将分类变量（如颜色、尺寸等）转换为数值型特征，便于机器学习模型处理。
# 具体做法是：对于每种可能的取值，创建一个新的二进制特征列，如果当前样本属于该取值则该列为1，否则为0。
# 例如“颜色”有“红”“蓝”“绿”，就变成“颜色=红”“颜色=蓝”“颜色=绿”三列，每行对应0或1。
# 这样可以避免分类变量本身的“大小关系”对模型造成误导，同时让模型能够感知不同类别的信息。

特征名字: ['价格' '尺寸=中' '尺寸=大' '尺寸=小' '颜色=红' '颜色=蓝']
转换后的数据:
 [[100.   0.   1.   0.   1.   0.]
 [ 80.   1.   0.   0.   0.   1.]
 [ 50.   0.   0.   1.   1.   0.]]


### CountVectorizer

In [7]:
import jieba
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "人生苦短，我喜欢 python",
    "人生漫长，不用 python python",
    "人生漫漫，不用 python python python"
]

# 用jieba分词
documents_cut = [" ".join(jieba.lcut(doc)) for doc in documents]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents_cut)

print("特征名称:", vectorizer.get_feature_names_out())
print("转换后的结果:\n", X.toarray())

Building prefix dict from the default dictionary ...
Dumping model to file cache /var/folders/y0/bfxm66992x3f3164jm5lmndm0000gn/T/jieba.cache
Loading model cost 0.324 seconds.
Prefix dict has been built successfully.


特征名称: ['python' '不用' '人生' '喜欢' '漫漫' '漫长' '苦短']
转换后的结果:
 [[1 0 1 1 0 0 1]
 [2 1 1 0 0 1 0]
 [3 1 1 0 1 0 0]]


### TfidfVectorizer

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    "I love machine learning. Machine learning is interesting.",
    "I love coding. Coding is fun and useful.",
    "Machine learning and coding are my favorites kills."
]

# 直接使用英文原文（没有额外分词）
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(documents)

print("特征名称:", vectorizer.get_feature_names_out())
print("TF-IDF矩阵:\n", X.toarray())



特征名称: ['and' 'are' 'coding' 'favorites' 'fun' 'interesting' 'is' 'kills'
 'learning' 'love' 'machine' 'my' 'useful']
TF-IDF矩阵:
 [[0.         0.         0.         0.         0.         0.3839346
  0.29199216 0.         0.58398432 0.29199216 0.58398432 0.
  0.        ]
 [0.30922846 0.         0.61845693 0.         0.40659827 0.
  0.30922846 0.         0.         0.30922846 0.         0.
  0.40659827]
 [0.30267425 0.39798027 0.30267425 0.39798027 0.         0.
  0.         0.39798027 0.30267425 0.         0.30267425 0.39798027
  0.        ]]


### 归一化

In [9]:
import numpy as np

# 用 numpy array 生成数组
arr = np.array([[1,200],[2,300],[3,400]])
print("原始数组：\n", arr)

# 归一化处理（按每列进行0-1归一化）
arr_min = arr.min(axis=0)
arr_max = arr.max(axis=0)
arr_normalized = (arr - arr_min) / (arr_max - arr_min)
print("归一化后的数组：\n", arr_normalized)

原始数组：
 [[  1 200]
 [  2 300]
 [  3 400]]
归一化后的数组：
 [[0.  0. ]
 [0.5 0.5]
 [1.  1. ]]


### 标准化

In [10]:
# 标准化处理（按每列做z-score标准化，即减均值除以标准差）
arr_mean = arr.mean(axis=0)
arr_std = arr.std(axis=0)
arr_standardized = (arr - arr_mean) / arr_std
print("标准化后的数组：\n", arr_standardized)

标准化后的数组：
 [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


In [11]:
# 计算标准化后数组每列的均值和标准差
mean_after_std = arr_standardized.mean(axis=0)
std_after_std = arr_standardized.std(axis=0)
print("标准化后各列的均值：", mean_after_std)
print("标准化后各列的标准差：", std_after_std)

# 经过标准化后的数组，每列的均值都会为0.，每列的标准差都会是1.


标准化后各列的均值： [0. 0.]
标准化后各列的标准差： [1. 1.]


In [12]:
from IPython.display import display, Math

# 标准化后均值为0的证明公式
mean_formula = r"""
\text{均值} = \frac{1}{n} \sum_{i=1}^n z_i = \frac{1}{n}\sum_{i=1}^n\frac{x_i - \mu}{\sigma} = \frac{1}{\sigma}\left( \frac{1}{n}\sum_{i=1}^n x_i - \mu \right)
= \frac{1}{\sigma}(\mu - \mu) = 0
"""

# 标准化后方差为1的证明公式
var_formula = r"""
\text{方差} = \frac{1}{n}\sum_{i=1}^n (z_i - 0)^2 = \frac{1}{n} \sum_{i=1}^n \left( \frac{x_i - \mu}{\sigma} \right)^2 = \frac{1}{\sigma^2} \frac{1}{n} \sum_{i=1}^n (x_i - \mu)^2
= \frac{1}{\sigma^2} \cdot \sigma^2 = 1
"""

display(Math(mean_formula))
display(Math(var_formula))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 特征选择

- 删除方差小的样本，这样的样本没有训练意义
- 方差为0，则数据都一样